# LSTM FAQ Chatbot — Complete Mini Project

A simple end-to-end **sequence-to-sequence LSTM chatbot**.

### What this notebook does

```text
FAQ dataset
    ↓
Data cleaning/checking
    ↓
Tokenization
    ↓
<START> / <END>
    ↓
Padding
    ↓
Encoder + Decoder LSTM
    ↓
Training
    ↓
Save model + tokenizer
    ↓
Inference
    ↓
Test chatbot
    ↓
API-ready artifacts
```

This notebook keeps the ML part simple so you can understand every step.

> **Important:** This is a learning project. A tiny FAQ dataset will not make a production-quality chatbot. The goal here is to understand how an LSTM sequence-to-sequence chatbot works.


## 1. Import libraries

We use:
- **NumPy** for numerical work
- **Pandas** for the FAQ dataset
- **TensorFlow/Keras** for the LSTM
- **Matplotlib** for training graphs
- **pickle/json** for saving supporting files


In [2]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


NumPy: 2.5.1
Pandas: 3.0.3
TensorFlow: 2.21.0
GPU: []


## 2. Set project paths

This notebook expects the same project structure you are already using:

```text
LSTM-Chatbot/
├── data/
│   └── faq.csv
├── notebooks/
│   └── 01_lstm_chatbot_clean.ipynb
└── models/
```

If this notebook is inside `notebooks/`, the paths below will work.


In [3]:
DATA_PATH = "../data/faq.csv"
MODEL_DIR = Path("../models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Model directory:", MODEL_DIR.resolve())


Model directory: F:\ML\AI_Engineering\AI-Engineering-Bootcamp\Projects\LSTM-Chatbot\models


## 3. Load the FAQ dataset

Your CSV should contain two columns:

```text
question,answer
```


In [4]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())


Dataset shape: (10, 2)


,question,answer
0,What is the course fee?,The course fee is 799 rupees per month.
1,How long is the course?,The course duration is 7 months.
2,What is the course syllabus?,The syllabus includes Python SQL data analysis...
3,Is deep learning included?,No deep learning is not included in the program.
4,Is NLP included?,No NLP is not included in the program.


## 4. Basic data checking

Before training, make sure the dataset does not contain missing questions or answers.


In [5]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("\nColumns:", df.columns.tolist())


Missing values:
question    0
answer      0
dtype: int64

Duplicate rows: 0

Columns: ['question', 'answer']


In [6]:
# Remove rows with missing text
df = df.dropna(subset=["question", "answer"]).copy()

# Remove exact duplicate question-answer pairs
df = df.drop_duplicates(subset=["question", "answer"]).reset_index(drop=True)

print("Final dataset shape:", df.shape)


Final dataset shape: (10, 2)


## 5. Inspect a few examples

In [7]:
for i in range(min(5, len(df))):
    print("QUESTION:", df.iloc[i]["question"])
    print("ANSWER:  ", df.iloc[i]["answer"])
    print("-" * 60)


QUESTION: What is the course fee?
ANSWER:   The course fee is 799 rupees per month.
------------------------------------------------------------
QUESTION: How long is the course?
ANSWER:   The course duration is 7 months.
------------------------------------------------------------
QUESTION: What is the course syllabus?
ANSWER:   The syllabus includes Python SQL data analysis and machine learning.
------------------------------------------------------------
QUESTION: Is deep learning included?
ANSWER:   No deep learning is not included in the program.
------------------------------------------------------------
QUESTION: Is NLP included?
ANSWER:   No NLP is not included in the program.
------------------------------------------------------------


## 6. Add `<START>` and `<END>` to every answer

The decoder needs two special signals:

- `<START>` = start generating an answer
- `<END>` = stop generating an answer

Example:

```text
The course lasts 7 months.
```

becomes:

```text
<START> The course lasts 7 months. <END>
```

We do this **before fitting the tokenizer** so the special tokens receive real IDs.


In [8]:
df["answer_with_tokens"] = (
    "<START> " + df["answer"].astype(str) + " <END>"
)

df[["answer", "answer_with_tokens"]].head()


,answer,answer_with_tokens
0,The course fee is 799 rupees per month.,<START> The course fee is 799 rupees per month...
1,The course duration is 7 months.,<START> The course duration is 7 months. <END>
2,The syllabus includes Python SQL data analysis...,<START> The syllabus includes Python SQL data ...
3,No deep learning is not included in the program.,<START> No deep learning is not included in th...
4,No NLP is not included in the program.,<START> No NLP is not included in the program....


## 7. Tokenization

The tokenizer converts words into integer IDs.

Example:

```text
What is SQL?
      ↓
[5, 3, 8]
```

Tokenization does **not** mean the model understands the sentence. It only gives every known word an ID.


In [9]:
tokenizer = Tokenizer(oov_token="<OOV>")

all_text = (
    df["question"].tolist()
    + df["answer_with_tokens"].tolist()
)

tokenizer.fit_on_texts(all_text)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

print("Vocabulary size:", vocab_size)
print("\nFirst 30 tokens:")
print(dict(list(word_index.items())[:30]))


Vocabulary size: 49

First 30 tokens:
{'<OOV>': 1, 'is': 2, 'the': 3, 'start': 4, 'end': 5, 'included': 6, 'course': 7, 'learning': 8, 'in': 9, 'program': 10, 'what': 11, 'sql': 12, 'machine': 13, 'yes': 14, 'fee': 15, 'how': 16, 'syllabus': 17, 'deep': 18, 'nlp': 19, 'programming': 20, 'language': 21, 'practical': 22, 'can': 23, 'enroll': 24, 'includes': 25, 'python': 26, 'and': 27, 'no': 28, 'not': 29, 'long': 30}


## 8. Get the special token IDs

Keras lowercases tokens by default, so `<START>` becomes `<start>` and `<END>` becomes `<end>`.


In [10]:
start_token = tokenizer.word_index["start"]
end_token = tokenizer.word_index["end"]

print("START token:", start_token)
print("END token:", end_token)
print("OOV token:", tokenizer.word_index["<OOV>"])


START token: 4
END token: 5
OOV token: 1


## 9. Convert questions and answers into sequences

Each sentence is now represented by numbers.


In [11]:
question_sequences = tokenizer.texts_to_sequences(
    df["question"].tolist()
)

answer_sequences = tokenizer.texts_to_sequences(
    df["answer_with_tokens"].tolist()
)

print("Question example:")
print(question_sequences[0])

print("\nAnswer example:")
print(answer_sequences[0])


Question example:
[11, 2, 3, 7, 15]

Answer example:
[4, 3, 7, 15, 2, 33, 34, 35, 36, 5]


## 10. Prepare decoder input and target

This is one of the most important ideas.

Suppose the answer is:

```text
<START> the fee is 799 <END>
```

The decoder input is:

```text
<START> the fee is 799
```

The target is:

```text
the fee is 799 <END>
```

So the model learns:

```text
<START> → the
the     → fee
fee     → is
is      → 799
799     → <END>
```


In [12]:
decoder_inputs = []
decoder_targets = []

for sequence in answer_sequences:
    decoder_inputs.append(sequence[:-1])
    decoder_targets.append(sequence[1:])

print("Decoder input:", decoder_inputs[0])
print("Decoder target:", decoder_targets[0])


Decoder input: [4, 3, 7, 15, 2, 33, 34, 35, 36]
Decoder target: [3, 7, 15, 2, 33, 34, 35, 36, 5]


## 11. Find maximum sequence lengths

The LSTM expects fixed-size tensors, so we find the longest question and longest decoder sequence.


In [13]:
max_question_length = max(
    len(sequence) for sequence in question_sequences
)

max_answer_length = max(
    len(sequence) for sequence in decoder_inputs
)

print("Maximum question length:", max_question_length)
print("Maximum answer length:", max_answer_length)


Maximum question length: 5
Maximum answer length: 11


## 12. Pad the sequences

Short sequences receive `0` at the end.

Example:

```text
[4, 7, 2]
→
[4, 7, 2, 0, 0]
```

`0` means padding, not a real word.


In [14]:
X_encoder = pad_sequences(
    question_sequences,
    maxlen=max_question_length,
    padding="post"
)

X_decoder = pad_sequences(
    decoder_inputs,
    maxlen=max_answer_length,
    padding="post"
)

y = pad_sequences(
    decoder_targets,
    maxlen=max_answer_length,
    padding="post"
)

print("Encoder:", X_encoder.shape)
print("Decoder:", X_decoder.shape)
print("Target:", y.shape)


Encoder: (10, 5)
Decoder: (10, 11)
Target: (10, 11)


## 13. Padding mask for the loss

Our target contains `0` values from padding.

We do **not** want the model to be rewarded for predicting padding.

So we create a weight:
- `1` for a real target token
- `0` for padding

This makes the training loss focus on real words.


In [15]:
sample_weights = (y != 0).astype("float32")

print("Example target:")
print(y[0])

print("\nExample weights:")
print(sample_weights[0])


Example target:
[ 3  7 15  2 33 34 35 36  5  0  0]

Example weights:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0.]


## 14. Build the Encoder

The encoder reads the question and produces two important states:

- `state_h`
- `state_c`

These states carry information from the question to the decoder.


In [16]:
embedding_dim = 128
lstm_units = 128

encoder_inputs = Input(
    shape=(max_question_length,),
    name="encoder_inputs"
)

encoder_embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="encoder_embedding"
)

encoder_embedding = encoder_embedding_layer(encoder_inputs)

encoder_lstm = LSTM(
    lstm_units,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(encoder_embedding)


## 15. Build the Decoder

The decoder receives:
- the previous answer token
- the encoder states

It then predicts the next token.


In [17]:
decoder_inputs = Input(
    shape=(max_answer_length,),
    name="decoder_inputs"
)

decoder_embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(decoder_inputs)

decoder_lstm = LSTM(
    lstm_units,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)


## 16. Output layer

For every decoder time step, the Dense layer gives a probability for every word in the vocabulary.

```text
LSTM output
    ↓
Dense + Softmax
    ↓
probability of every possible word
```


In [18]:
decoder_dense = Dense(
    vocab_size,
    activation="softmax",
    name="decoder_output"
)

decoder_outputs = decoder_dense(decoder_outputs)


## 17. Create the complete training model

In [19]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs,
    name="lstm_faq_chatbot"
)

model.summary()


Model: "lstm_faq_chatbot"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 11)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 5, 128)    │      6,272 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 5)         │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 11, 128)   │      6,272 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │    131,584 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 11, 128), │    131,584 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 11, 49)    │      6,321 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 282,033 (1.08 MB)

 Trainable params: 282,033 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

## 18. Compile the model

We use:
- **Adam** optimizer
- **Sparse categorical cross-entropy** because targets are integer token IDs

The padding weights will be supplied during training.


In [20]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


## 19. Train the chatbot

For this small dataset, start with 100 epochs.

Because the dataset is tiny, the model may overfit. That is expected for this learning project.

Watch the loss and validation loss instead of assuming more epochs always means a better chatbot.


In [21]:
history = model.fit(
    [X_encoder, X_decoder],
    y,
    sample_weight=sample_weights,
    epochs=100,
    batch_size=4,
    validation_split=0.2,
    shuffle=True
)


Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 815ms/step - accuracy: 0.0571 - loss: 3.8887 - val_accuracy: 0.1111 - val_loss: 3.8890
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - accuracy: 0.2714 - loss: 3.8696 - val_accuracy: 0.1667 - val_loss: 3.8852
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - accuracy: 0.2714 - loss: 3.8472 - val_accuracy: 0.1111 - val_loss: 3.8798
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.2714 - loss: 3.8203 - val_accuracy: 0.1111 - val_loss: 3.8724
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.2714 - loss: 3.7863 - val_accuracy: 0.1111 - val_loss: 3.8611
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.2714 - loss: 3.7310 - val_accuracy: 0.1111 - val_loss: 3.8426
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.2571 - loss: 3.6454 - val_accuracy: 0.1111 - val_loss: 3.8107
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.2143 - loss: 3.5082 - val_accuracy: 0.1111 - val_lo

## 20. Plot training history

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Chatbot Training")
plt.legend()
plt.show()


## 21. Save the trained model and tokenizer

These files are what the later API will use.

We save:
- training model
- tokenizer
- configuration


In [22]:
model.save(MODEL_DIR / "lstm_chatbot.keras")

with open(MODEL_DIR / "tokenizer.pkl", "wb") as file:
    pickle.dump(tokenizer, file)

config = {
    "vocab_size": int(vocab_size),
    "embedding_dim": int(embedding_dim),
    "lstm_units": int(lstm_units),
    "max_question_length": int(max_question_length),
    "max_answer_length": int(max_answer_length),
    "start_token": int(start_token),
    "end_token": int(end_token)
}

with open(MODEL_DIR / "config.json", "w") as file:
    json.dump(config, file, indent=4)

print("Saved:")
print(MODEL_DIR / "lstm_chatbot.keras")
print(MODEL_DIR / "tokenizer.pkl")
print(MODEL_DIR / "config.json")


Saved:
..\models\lstm_chatbot.keras
..\models\tokenizer.pkl
..\models\config.json


# 22. Inference

Training is finished.

Now we build the models used when a real user asks a question.

Training:

```text
question + correct previous answer words
                    ↓
                  model
```

Inference:

```text
question
   ↓
encoder
   ↓
decoder
   ↓
one word
   ↓
feed that word back
   ↓
next word
   ↓
...
   ↓
<END>
```


## 23. Build the encoder inference model

It takes a question and returns the encoder states.


In [29]:
encoder_model = Model(
    encoder_inputs,
    [state_h, state_c],
    name="encoder_inference"
)


## 24. Build the decoder inference model

During inference, the decoder receives only **one token at a time**.


In [30]:
decoder_token_input = Input(
    shape=(1,),
    name="decoder_token_input"
)

decoder_state_h_input = Input(
    shape=(lstm_units,),
    name="decoder_state_h_input"
)

decoder_state_c_input = Input(
    shape=(lstm_units,),
    name="decoder_state_c_input"
)

decoder_token_embedding = decoder_embedding_layer(
    decoder_token_input
)

decoder_token_output, decoder_state_h, decoder_state_c = decoder_lstm(
    decoder_token_embedding,
    initial_state=[
        decoder_state_h_input,
        decoder_state_c_input
    ]
)

decoder_token_output = decoder_dense(decoder_token_output)

decoder_model = Model(
    [
        decoder_token_input,
        decoder_state_h_input,
        decoder_state_c_input
    ],
    [
        decoder_token_output,
        decoder_state_h,
        decoder_state_c
    ],
    name="decoder_inference"
)

decoder_model.summary()


Model: "decoder_inference"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ decoder_token_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 1, 128)    │      6,272 │ decoder_token_in… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_state_h_in… │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_state_c_in… │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 1, 128),  │    131,584 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ decoder_state_h_… │
│                     │ (None, 128)]      │            │ decoder_state_c_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output      │ (None, 1, 49)     │      6,321 │ decoder_lstm[2][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 144,177 (563.19 KB)

 Trainable params: 144,177 (563.19 KB)

 Non-trainable params: 0 (0.00 B)

## 25. Create number → word mapping

The model predicts an integer.

We convert that integer back into a word.


In [31]:
index_to_word = {
    index: word
    for word, index in tokenizer.word_index.items()

}

print(list(index_to_word.items())[:10])


[(1, '<OOV>'), (2, 'is'), (3, 'the'), (4, 'start'), (5, 'end'), (6, 'included'), (7, 'course'), (8, 'learning'), (9, 'in'), (10, 'program')]


## 26. Create `generate_response()`

This is the main chatbot function.

It:
1. tokenizes the question
2. pads it
3. encodes it
4. starts with `<START>`
5. predicts one token
6. feeds that token back
7. stops at `<END>`


In [32]:
def generate_response(question):
    # Convert question to token IDs
    question_sequence = tokenizer.texts_to_sequences([question])

    # Pad the question
    question_sequence = pad_sequences(
        question_sequence,
        maxlen=max_question_length,
        padding="post"
    )

    # Encode the question
    state_h, state_c = encoder_model.predict(
        question_sequence,
        verbose=0
    )

    # Start decoder with <START>
    target_token = np.array([[start_token]])

    response_words = []

    for _ in range(max_answer_length):
        output_tokens, state_h, state_c = decoder_model.predict(
            [
                target_token,
                state_h,
                state_c
            ],
            verbose=0
        )

        # Most probable next token
        predicted_token = int(
            np.argmax(output_tokens[0, -1, :])
        )

        # Stop when <END> is predicted
        if predicted_token == end_token:
            break

        predicted_word = index_to_word.get(
            predicted_token,
            ""
        )

        if predicted_word and predicted_word not in ["<start>", "<end>"]:
            response_words.append(predicted_word)

        # Feed prediction back into decoder
        target_token = np.array([[predicted_token]])

    return " ".join(response_words)


## 27. Test one question

In [33]:
question = "What is the course fee?"

response = generate_response(question)

print("User:", question)
print("Bot:", response)


User: What is the course fee?
Bot: the course fee is 799 rupees per month


## 28. Test several questions

Test both questions that look like your training data and questions with different wording.


In [34]:
test_questions = [
    "What is the course fee?",
    "How long is the course?",
    "Is SQL included?",
    "Is deep learning included?",
    "What programming language is used?",
    "How much does the course cost?"
]

for question in test_questions:
    response = generate_response(question)

    print("User:", question)
    print("Bot: ", response)
    print("-" * 60)


User: What is the course fee?
Bot:  the course fee is 799 rupees per month
------------------------------------------------------------
User: How long is the course?
Bot:  the course duration is 7 months
------------------------------------------------------------
User: Is SQL included?
Bot:  yes sql is included in the program
------------------------------------------------------------
User: Is deep learning included?
Bot:  no deep learning is not included in the program
------------------------------------------------------------
User: What programming language is used?
Bot:  python is the main programming language
------------------------------------------------------------
User: How much does the course cost?
Bot:  the includes python data analysis and machine learning
------------------------------------------------------------


## 29. Interactive chatbot

Type `exit` to stop.


In [ ]:
while True:
    question = input("You: ").strip()

    if question.lower() == "exit":
        print("Bot: Goodbye!")
        break

    if not question:
        continue

    response = generate_response(question)

    print("Bot:", response)


# 30. Save inference models for the API

The API should not need to retrain the chatbot.

We save the encoder and decoder inference models separately.

Later:

```text
FastAPI
   ↓
load encoder
load decoder
load tokenizer
load config
   ↓
/chat
   ↓
generate response
```


In [ ]:
encoder_model.save(MODEL_DIR / "encoder_model.keras")
decoder_model.save(MODEL_DIR / "decoder_model.keras")

print("Inference models saved.")


# 31. Final model files

After running the notebook, your `models/` folder should contain:

```text
models/
├── lstm_chatbot.keras
├── encoder_model.keras
├── decoder_model.keras
├── tokenizer.pkl
└── config.json
```

These are the artifacts we need for the next engineering stage.

### Next stage

Do **not** build the frontend yet.

First build:

```text
FastAPI
   ↓
POST /chat
   ↓
LSTM inference
   ↓
JSON response
```

Then connect a simple frontend chat UI to the API.


# 32. Important learning notes

### What you implemented

You built a real sequence-to-sequence LSTM pipeline:

```text
Text
 ↓
Tokenizer
 ↓
Integer sequences
 ↓
Padding
 ↓
Embedding
 ↓
Encoder LSTM
 ↓
Hidden states
 ↓
Decoder LSTM
 ↓
Softmax
 ↓
Next-token prediction
```

### What to experiment with later

Change one thing at a time:

- number of FAQ examples
- `embedding_dim`
- `lstm_units`
- number of epochs
- batch size
- question wording
- answer length

Then observe how the chatbot changes.

### Limitation

A tiny FAQ dataset is not enough for a robust conversational assistant. This project is mainly for learning **tokenization → sequence preparation → LSTM encoder-decoder → teacher forcing → autoregressive inference → model serving**.
